# Healthcare Exploratory Data Analysis

**Dataset:** Diabetes 130-US Hospitals for Years 1999-2008
**Objective:** Perform a comprehensive Exploratory Data Analysis (EDA) on a real-world healthcare dataset to profile the data, identify missing values and outliers, and evaluate suitable machine learning problem types.

## SECTION 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## SECTION 2 — Load Dataset

In [ ]:
# Load the dataset
# Note: In this dataset, missing values are represented by '?'
df = pd.read_csv('../dataset/dataset_diabetes/diabetic_data.csv', na_values='?')

print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

## SECTION 3 — Dataset Overview

In [ ]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"Dataset shape: {df.shape}\n")

print("Dataset Info (Column names, Data types, Memory usage):")
df.info(memory_usage='deep')

## SECTION 4 — Data Type Analysis

- **Numerical features** are crucial for statistical modeling, correlation analysis, and regression tasks. They represent measurable quantities.
- **Categorical features** represent distinct groups or classes. They are essential for classification tasks and understanding patient demographics or clinical categories. They often require encoding before modeling.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Features ({len(numerical_cols)}):\n{numerical_cols}\n")
print(f"Categorical Features ({len(categorical_cols)}):\n{categorical_cols}")

## SECTION 5 — Missing Value Analysis

In healthcare data, missing values are very common. A missing 'weight' might mean it wasn't recorded or wasn't considered relevant for that specific admission. Missing 'medical_specialty' often indicates the admitting physician was a generalist or the specialty wasn't documented in the EHR system.

**Handling Approaches:**
- **High missingness (>80%)**: Features like 'weight' may need to be dropped unless their missingness is predictive itself.
- **Categorical missingness**: Can be treated as a separate category (e.g., 'Unknown').

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

missing_data = pd.DataFrame({'Missing Values': missing_counts, 'Percentage (%)': missing_percentages})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)

display(missing_data)

# Visualization
plt.figure(figsize=(10, 5))
sns.barplot(x=missing_data['Percentage (%)'], y=missing_data.index, palette='viridis')
plt.title('Percentage of Missing Values per Column')
plt.xlabel('Percentage (%)')
plt.ylabel('Features')
plt.show()

## SECTION 6 — Duplicate Analysis

If entire rows are exact duplicates, it often indicates a data-entry error or a flaw in the database extraction process. These should usually be removed to prevent bias. However, in this dataset, we expect 0 full row duplicates because each encounter has a unique `encounter_id`.

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Number of duplicate rows: {duplicate_count}")
print(f"Percentage of duplicate rows: {duplicate_percentage:.4f}%")

## SECTION 7 — Descriptive Statistics

In [ ]:
print("--- Numerical Descriptive Statistics ---")
display(df[numerical_cols].describe().T)

print("\n--- Categorical Descriptive Statistics ---")
display(df[categorical_cols].describe().T)

## SECTION 8 — Numerical Feature Analysis

In [ ]:
important_num = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(important_num):
    sns.histplot(df[col], kde=True, ax=axes[i], bins=30)
    axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

**Observations:**
- `time_in_hospital` is heavily right-skewed, meaning most patients stay for a few days, but a long tail exists for severe cases.
- `num_lab_procedures` follows a somewhat normal distribution but is slightly right-skewed.
- `num_medications` is right-skewed, showing most patients receive 10-20 medications, but some receive over 60.

## SECTION 9 — Categorical Feature Analysis

In [ ]:
important_cat = ['race', 'gender', 'age', 'admission_type_id']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(important_cat):
    sns.countplot(y=df[col], ax=axes[i], order=df[col].value_counts().index)
    axes[i].set_title(f'Value Counts for {col}')

plt.tight_layout()
plt.show()

**Observations:**
- `age` shows that the vast majority of diabetic patients in this dataset are between 50 and 90 years old.
- `race` is heavily imbalanced, primarily composed of Caucasian patients.
- `gender` is fairly balanced between Male and Female.

## SECTION 10 — OUTLIER ANALYSIS

> **Important Note:** In healthcare data, statistical outliers (values far from the mean/median) often represent **genuine severe medical cases** rather than data-entry errors. For instance, a patient with 80 medications or a 14-day hospital stay is clinically plausible for complex diabetes cases. We will identify them here using the Interquartile Range (IQR) method, but they should **not** be automatically removed before modeling.

In [ ]:
outlier_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(df)) * 100
    
    print(f"{col}:")
    print(f"  - Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"  - Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
    print(f"  - Potential Outliers: {len(outliers)} ({outlier_pct:.2f}%)\n")

# Visualize outliers with Boxplots
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[outlier_cols], orient="h")
plt.title('Boxplots for Outlier Detection')
plt.show()

## SECTION 11 — Correlation Analysis

In [ ]:
# Select a subset of numerical columns for correlation to avoid noise from ID columns
corr_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses', 'number_inpatient']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

**Observations:**
- `time_in_hospital` has a moderate positive correlation with `num_medications` (0.47). This makes sense: longer stays generally require more medical interventions.
- We must remember that **correlation does not imply causation**. A longer hospital stay does not necessarily *cause* more medications; both could be caused by the underlying severity of the patient's illness.

## SECTION 12 — Target Variable Analysis

Our target variable is `readmitted`. It indicates if a patient was readmitted to the hospital within 30 days (`<30`), after 30 days (`>30`), or not at all (`NO`).

In [ ]:
target_counts = df['readmitted'].value_counts()
target_pct = df['readmitted'].value_counts(normalize=True) * 100

print("Target Variable Distribution:")
for cat, count in target_counts.items():
    print(f"  - {cat}: {count} ({target_pct[cat]:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(x='readmitted', data=df, order=['NO', '>30', '<30'], palette='Set2')
plt.title('Distribution of Readmission (Target Variable)')
plt.ylabel('Count')
plt.show()

**Observation:**
The dataset is **imbalanced**. More than half (53.91%) of patients are not readmitted. The critical class (`<30` days) only represents 11.16% of the data. This class imbalance is vital to address during future ML modeling (e.g., using SMOTE or class weights), otherwise, a model might just predict 'NO' for everyone and achieve high misleading accuracy.

## SECTION 13 — Feature Relationships

In [ ]:
# Previous admissions vs readmission
plt.figure(figsize=(8, 5))
sns.boxplot(x='readmitted', y='number_inpatient', data=df, order=['NO', '>30', '<30'])
plt.title('Number of Previous Inpatient Visits vs. Readmission Status')
plt.ylim(0, 10)  # Zooming in for better visibility as there are large outliers
plt.show()

**Observation:** Patients readmitted within 30 days (`<30`) tend to have a higher median number of prior inpatient visits compared to those not readmitted.

## SECTION 14 — KEY EDA FINDINGS

1. **Dataset Size:** The dataset contains 101,766 records and 50 features.
2. **Missing Values:** 7 columns contain missing data. The `weight` feature has the highest percentage (96.86% missing), followed by `max_glu_serum` (94.75%) and `A1Cresult` (83.28%).
3. **Outliers:** Features like `num_medications` contain 2,557 potential outliers, and `time_in_hospital` contains 2,252 potential outliers. These likely represent complex medical cases.
4. **Class Imbalance:** The target variable (`readmitted`) is highly imbalanced, with 'NO' at 53.91% and '<30' at 11.16%.
5. **Demographics:** The data heavily skews older (most patients 50-90 years old) and Caucasian.
6. **Relationships:** A longer hospital stay is positively correlated with the number of medications administered.

## SECTION 15 — POSSIBLE MACHINE LEARNING PROBLEM TYPES

**A. CLASSIFICATION**
- **Supported?** Yes, highly supported.
- **Target:** `readmitted` (Multiclass: NO, >30, <30. Alternatively, Binary: Readmitted vs Not Readmitted).
- **Features:** Age, diagnoses codes, number of medications, prior inpatient visits, time in hospital.
- **Why it's appropriate:** Predicting hospital readmission is a classic classification problem used by hospitals to identify high-risk patients and intervene early, improving care and reducing healthcare costs.

**B. REGRESSION**
- **Supported?** Yes.
- **Target:** `time_in_hospital` (Length of Stay).
- **Features:** Age, admission type, number of diagnoses, emergency visits.
- **Why it's appropriate:** Predicting length of stay helps hospitals allocate beds and staff resources efficiently.

**C. CLUSTERING**
- **Supported?** Yes.
- **Target:** None.
- **Features:** Laboratory results, medication patterns, age groups.
- **Why it's appropriate:** Even without predicting readmission, unsupervised clustering can segment patients into meaningful subgroups (e.g., finding clusters of patients with specific comorbidities who share similar medication profiles). This can help in designing targeted care pathways.

## SECTION 16 — LIMITATIONS AND ETHICAL CONSIDERATIONS

- **Patient Privacy:** Though de-identified, high-dimensional healthcare data always carries a small risk of re-identification. Models must be built respecting HIPAA and patient confidentiality.
- **Dataset Bias:** The dataset is heavily skewed towards Caucasian populations. An ML model trained on this might perform poorly or unfairly on minority populations.
- **Missing Medical Records:** A lack of documentation (like missing `weight` or `medical_specialty`) doesn't mean the data doesn't exist, just that it wasn't captured in the EHR. Missingness can introduce severe bias.
- **Correlation vs. Causation:** We observed correlation between time in hospital and medications. The model might learn this, but it cannot infer that reducing medications will magically shorten a hospital stay.
- **AI as Support:** ML predictions (like readmission risk) should be used as clinical decision support tools to *assist* doctors, not replace their medical judgment.

## SECTION 17 — CONCLUSION

This Exploratory Data Analysis successfully profiled the Diabetes 130-US Hospitals dataset. We uncovered critical data quality issues, notably severe missing values in specific lab/measurement columns and significant class imbalance in the target variable. We identified that statistical outliers likely represent medically complex cases rather than simple errors. 

The dataset is extremely well-suited for a Classification model to predict patient readmissions, provided that class imbalance and data imputation are handled carefully during the preprocessing phase. Understanding these underlying patterns and limitations is an essential prerequisite before any machine learning model development begins.